# fedOPD · Exploratory Data Analysis
**Cross-silo federated forecasting of NHS outpatient (RTT) demand — operations research, not clinical.**

EDA on NHS England **Referral-to-Treatment** data: 96 monthly files, `2018-04 → 2026-03` (8 financial years).
Goal — verify the raw data, build the canonical **`[date, branch, dept, demand]`** table, and characterise it
enough to (a) select federated clients and (b) design the forecaster.

- **Grain:** trust × treatment function × month.
- **Demand:** `RTT Part Type == Part_3` (new RTT periods), column `Total All`.
- **Client:** one NHS trust (3-char `R##` provider code); independents (5-char) excluded.
- **Hierarchy:** NHS England → region → ICB (`Provider Parent`) → trust → department.

This **single notebook** runs stages 2 → 8 in sequence. Each stage ends with quantified findings (`F-xx`)
and decisions (`D-xx`) that are copied into `docs/WORKLOG.md`. Derived tables → `reports/tables/`,
figures → `reports/figures/`. Raw `data/` is read-only.

## Stage 2 · Input & inspect

**Questions this stage answers:**
(a) all 96 files exist and read · (b) the 9 canonical columns are present in every era ·
(c) the exact `Part_3` demand label · (d) trust vs independent provider-code split ·
(e) `C_999` aggregate specialty present · (f) `Period` parses (incl. the `APRIL`/`April` case quirk) ·
(g) `Total All` is numeric · (h) national totals match externally-verified figures ·
(i) schema is consistent across all 96 files.

**Sentinel months** (develop on 4; full 96-file passes only where marked): `2018-04` oldest / first modern
layout (69 cols) · `2020-04` COVID collapse · `2021-04` layout step to 121 cols · `2026-03` newest.

**Done when** `D-01` (loader `usecols` + parse rules) is logged.

In [9]:
import sys , platform , datetime as dt
import pandas as pd , numpy as np


print(dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "|", platform.node())

print("username : Nilesh")

print("python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)

2026-07-11 03:18:11 | ubuntu-Standard-PC-Q35-ICH9-2009
username : Nilesh
python 3.10.12 | pandas 2.3.3 | numpy 2.2.6


In [10]:
import os , shutil , subprocess , platform 

print("System Specifications")

cpu = platform.processor()
if shutil.which("lscpu"):                                            # lscpu gives the proper model name
    out = subprocess.run(["lscpu"], capture_output=True, text=True).stdout
    cpu = next((l.split(":",1)[1].strip() for l in out.splitlines()
                if l.startswith("Model name")), cpu)                 # grab the "Model name" line
print(f"CPU: {cpu} | {os.cpu_count()} cores")   

ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
print(f"RAM: {ram_gb:.0f} GB")

if shutil.which("nvidia-smi"):
    q = subprocess.run(["nvidia-smi",
 "--query-gpu=name,memory.total,memory.used",
 "--format=csv,noheader"], capture_output=True, text=True)
    print("GPU:" , q.stdout.strip())


System Specifications
CPU: Intel Xeon Processor (Cascadelake) | 42 cores
RAM: 67 GB
GPU: NVIDIA A100 80GB PCIe, 81920 MiB, 1 MiB


In [11]:
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "CLAUDE.md").exists())
DATA = ROOT / "data" / "nhs_rtt_referral_to_treatment"
files = sorted(DATA.glob("rtt_*.csv"))

print(len(files), "files:", files[0].name, "→", files[-1].name) 
sizes = [f.stat().st_size/1e6 for f in files]
print(f"MB: min {min(sizes):.0f} max {max(sizes):.0f}")
assert len(files) == 96  

96 files: rtt_2018-04.csv → rtt_2026-03.csv
MB: min 42 max 114


In [12]:
print("Sentinel Months placed as markers")
Sentinels = ["2018-04", "2020-04", "2021-04", "2026-03"]       # sentinels: oldest, COVID crash, mid, newest
df = {m: pd.read_csv(DATA/f"rtt_{m}.csv",              # read each sentinel into a dict of DataFrames
                     low_memory=False) for m in Sentinels}     # low_memory=False: infer dtypes from whole file
for m in Sentinels: print(m, df[m].shape)                      # rows × columns per era (expect 69 vs 121 cols)

Sentinel Months placed as markers
2018-04 (196340, 69)
2020-04 (125460, 69)
2021-04 (160251, 121)
2026-03 (183400, 121)


In [13]:
print("All columns present (schema grows 69 -> 121 across the window)")
new = df["2026-03"].columns                                   # newest era is the superset (121 cols)
old = set(df["2018-04"].columns)                              # oldest era (69 cols)
for i, c in enumerate(new, 1):
    print(f"{i:>3}. {c}" + ("" if c in old else "   (2021+ only)"))  # flag columns added later

All columns present (schema grows 69 -> 121 across the window)
  1. Period
  2. Provider Parent Org Code
  3. Provider Parent Name
  4. Provider Org Code
  5. Provider Org Name
  6. Commissioner Parent Org Code
  7. Commissioner Parent Name
  8. Commissioner Org Code
  9. Commissioner Org Name
 10. RTT Part Type
 11. RTT Part Description
 12. Treatment Function Code
 13. Treatment Function Name
 14. Gt 00 To 01 Weeks SUM 1
 15. Gt 01 To 02 Weeks SUM 1
 16. Gt 02 To 03 Weeks SUM 1
 17. Gt 03 To 04 Weeks SUM 1
 18. Gt 04 To 05 Weeks SUM 1
 19. Gt 05 To 06 Weeks SUM 1
 20. Gt 06 To 07 Weeks SUM 1
 21. Gt 07 To 08 Weeks SUM 1
 22. Gt 08 To 09 Weeks SUM 1
 23. Gt 09 To 10 Weeks SUM 1
 24. Gt 10 To 11 Weeks SUM 1
 25. Gt 11 To 12 Weeks SUM 1
 26. Gt 12 To 13 Weeks SUM 1
 27. Gt 13 To 14 Weeks SUM 1
 28. Gt 14 To 15 Weeks SUM 1
 29. Gt 15 To 16 Weeks SUM 1
 30. Gt 16 To 17 Weeks SUM 1
 31. Gt 17 To 18 Weeks SUM 1
 32. Gt 18 To 19 Weeks SUM 1
 33. Gt 19 To 20 Weeks SUM 1
 34. Gt 20 To 21 Weeks

In [14]:
print("Canonical Months that are most useful")
CANON = ["Period","Provider Org Code","Provider Org Name",        # the 9 columns the loader
         "Treatment Function Code","Treatment Function Name",     # will depend on
         "RTT Part Type","Total All"]

for m in Sentinels:
    print(m, "missing:", [c for c in CANON if c not in df[m].columns])  # any canonical col absent?
    print("  parent cols:", [c for c in df[m].columns if "Parent" in c]) # exact Provider Parent headers

Canonical Months that are most useful
2018-04 missing: []
  parent cols: ['Provider Parent Org Code', 'Provider Parent Name', 'Commissioner Parent Org Code', 'Commissioner Parent Name']
2020-04 missing: []
  parent cols: ['Provider Parent Org Code', 'Provider Parent Name', 'Commissioner Parent Org Code', 'Commissioner Parent Name']
2021-04 missing: []
  parent cols: ['Provider Parent Org Code', 'Provider Parent Name', 'Commissioner Parent Org Code', 'Commissioner Parent Name']
2026-03 missing: []
  parent cols: ['Provider Parent Org Code', 'Provider Parent Name', 'Commissioner Parent Org Code', 'Commissioner Parent Name']


In [ ]:
# C7 · part-type labels per era — the demand filter string ("Part_3") must be exact
#      and present in every era; note where "Part_2A" (decision-to-admit) first appears.
for m in Sentinels:
    print(m, sorted(df[m]["RTT Part Type"].astype(str).unique()))   # distinct pathway labels

In [ ]:
# C8 · trust vs independent provider split.
#      NHS trusts = 3-char codes starting "R"; independents = 5-char codes.
#      Print the code-length histogram + the trust count vs all providers, per era.
for m in Sentinels:
    c = df[m]["Provider Org Code"].astype(str).str.strip()            # provider codes, stripped
    tr = c[(c.str.len() == 3) & c.str.upper().str.startswith("R")].nunique()  # R## trusts
    print(m, dict(c.str.len().value_counts().sort_index()),           # {code_length: count}
          "| R## trusts:", tr, "of", c.nunique(), "providers")

In [ ]:
# C9 · distinct treatment functions per era, and whether the C_999 aggregate row
#      ("Total") is present (it must be, so the loader can drop it).
for m in Sentinels:
    tf = df[m][["Treatment Function Code", "Treatment Function Name"]].drop_duplicates()
    print(m, len(tf), "functions | C_999 present:",
          bool((tf["Treatment Function Code"] == "C_999").any()))

In [ ]:
# C10 · Period string format + case quirk. Early files say "RTT-APRIL-2018"
#       (upper), later ones "RTT-March-2026" (title). A case-insensitive regex must
#       recover the month, and the title-cased month must equal the filename month.
import re
for m in Sentinels:
    p = str(df[m]["Period"].iloc[0])                       # Period string, first row
    g = re.search(r"([A-Za-z]+)[-_ ](\d{4})", p)           # month word + 4-digit year
    print(m, repr(p), "->", g.group(1).title(), g.group(2))  # normalised month + year

In [ ]:
# C11 · demand column dtype + NaN count per era. If any era is `object`, the loader
#       must coerce with pd.to_numeric(errors="coerce").
for m in Sentinels:
    col = df[m]["Total All"]                               # the demand column
    print(m, col.dtype, "| NaN:", int(pd.isna(col).sum()))

In [ ]:
# C12 · external validation (catches any wrong filter). Apply the full demand rule
#       — Part_3 · NHS trusts (R##) · drop C_999 — sum Total All, compare to the
#       verified national figures. Any MISMATCH => a filter is wrong; do NOT proceed.
EXPECT = {"2018-04": 1_544_011, "2020-04": 474_295,       # verified national Part_3
          "2021-04": 1_394_578, "2026-03": 1_783_457}      # totals (trusts, no C_999)

for m in Sentinels:
    d = df[m]
    c = d["Provider Org Code"].astype(str).str.strip()
    mask = ((d["RTT Part Type"] == "Part_3")                       # demand rows
            & (d["Treatment Function Code"] != "C_999")            # drop aggregate specialty
            & (c.str.len() == 3) & c.str.upper().str.startswith("R"))  # NHS trusts only
    tot = int(pd.to_numeric(d.loc[mask, "Total All"], errors="coerce").sum())
    print(m, f"{tot:,}", "vs", f"{EXPECT[m]:,}",
          "OK" if tot == EXPECT[m] else "MISMATCH")

In [ ]:
# C13 · schema audit across ALL 96 files (reads only the header row of each -> fast).
#       Confirms the 7 canonical columns exist everywhere, pins the exact month the
#       column count steps 69 -> 121, and checks the parent header is one variant.
import csv
rows = []
for f in files:
    with open(f, newline="") as fh:
        head = next(csv.reader(fh))                                # header row only
    rows.append({"month": f.stem[-7:],                             # YYYY-MM
                 "ncols": len(head),                               # column count
                 "canon_ok": all(c in head for c in CANON),        # 7 canonical present?
                 "parent": next((c for c in head                   # first Provider/Commissioner
                                  if "Parent" in c and "Code" in c), None)})  # parent code header
aud = pd.DataFrame(rows)
print("all canonical present:", bool(aud.canon_ok.all()))          # must be True
print(aud.groupby("ncols").month.agg(["min", "max", "count"]))     # exact 69->121 boundary
print("parent header variants:", list(aud.parent.unique()))        # expect one name
assert aud.canon_ok.all()

In [ ]:
# C14 · content sweep across ALL 96 files (reads only the 2 columns that matter).
#       Confirms Part_3 exists and Total All is numeric (<1% blanks tolerated) everywhere.
#       This is the one slow full-data pass in Stage 2 (~1-3 min).
probs = []                                                          # (file, reason) problems
for f in files:
    d = pd.read_csv(f, usecols=["RTT Part Type", "Total All"], low_memory=False)
    if "Part_3" not in set(d["RTT Part Type"].astype(str).unique()):
        probs.append((f.stem, "no Part_3"))                        # demand label must exist
    bad = pd.to_numeric(d["Total All"], errors="coerce").isna().mean()
    if bad > 0.01:                                                  # tolerate <1% blanks
        probs.append((f.stem, f"Total All non-numeric {bad:.1%}"))
print(len(probs), "problem files")
probs[:10]

### Stage 2 · findings & decision  →  copy to `docs/WORKLOG.md`

- **F-01** · Column layout steps **69 → 121 at `2021-04`** (36 months at 69 cols, 60 at 121); wait-bands extended 52 → 104+ weeks (post-COVID backlog becoming visible in the schema). Treatment functions grew **20 → 24**; `C_999` aggregate present in every era.
- **F-02** · `Part_3` demand label confirmed in all eras; the part-type set `{Part_1A, Part_1B, Part_2, Part_2A, Part_3}` is **stable** — `Part_2A` present from `2018-04`, nothing added later.
- **F-03** · NHS trusts (`R##`): **184 · 173 · 170 · 149** across the sentinels, of **417 / 414 / 431 / 533** providers. Provider-code lengths are strictly `{3, 5}`. Trust count **shrinks** (mergers) while total providers **grow** (independent sector).
- **F-04** · `Period` case is non-monotonic (`April` / `APRIL` / `APRIL` / `March`) → parse **case-insensitively**; title-cased month matches the filename **4/4**.
- **F-05** · `Total All` dtype is `int64` in every era; **0 NaN**.
- **F-06** · National `Part_3` totals match the verified figures **4/4**; the COVID collapse is real — `2020-04 = 474,295 ≈ 31%` of the `2018-04` baseline (`1,544,011`).
- **F-07** · Parent header is stable across all 96 files: **`Provider Parent Org Code`** (single variant) → no era-branching needed in `usecols`.
- **F-08** · `Part_3` + numeric `Total All` present in **96/96** files (0 problem files).

**D-01 — canonical loader spec.** `usecols` = the **9 canonical columns** (7 keys/filter/measure + `Provider Parent Org Code` / `Provider Parent Name` for the ICB tier); parse `Period` **case-insensitively**; demand = `Part_3` · NHS trusts (`R##`) · drop `C_999` · **sum over commissioners** · `to_numeric(errors="coerce")`.

## Stage 2b · Canonical build  (executes D-01)

Turn the 96 raw CSVs into one tidy table **`[date, branch, branch_name, icb, dept, dept_name, demand]`**
— one row per **trust × department × month** — with a full filter **ledger** (nothing dropped silently)
and a re-validation against the verified national totals.

Output: **`reports/tables/rtt_canonical.parquet`** — the single source every later stage loads
(the raw CSVs are never read again).

**Done when** the parquet is saved, the ledger is logged (`F-09`), and the asserts pass.

In [ ]:
# Stage 2b · C1 — canonical loader (executes D-01). Reads ONLY the 9 canonical columns.
import re
OUT = ROOT / "reports" / "tables"; OUT.mkdir(parents=True, exist_ok=True)   # derived-table dir

USECOLS = ["Period", "Provider Org Code", "Provider Org Name",
           "Provider Parent Org Code", "Provider Parent Name",
           "Treatment Function Code", "Treatment Function Name",
           "RTT Part Type", "Total All"]                    # keys + filter + measure + ICB tier

MONTHS = {m.upper(): i for i, m in enumerate(               # month name -> number (case-insensitive)
    ["January", "February", "March", "April", "May", "June", "July",
     "August", "September", "October", "November", "December"], start=1)}

def parse_period(p):                                        # 'RTT-APRIL-2020' -> Timestamp(2020-04-01)
    g = re.search(r"([A-Za-z]+)[-_ ](\d{4})", str(p))       # month word + year
    return pd.Timestamp(int(g.group(2)), MONTHS[g.group(1).upper()], 1)

def load_month(path, ledger):                               # one CSV -> canonical rows, updating the ledger
    d = pd.read_csv(path, usecols=USECOLS, low_memory=False)
    ledger["raw"] += len(d)                                 # 0. raw rows read
    d = d[d["RTT Part Type"] == "Part_3"]                   # 1. demand rows only
    ledger["part3"] += len(d)
    c = d["Provider Org Code"].astype(str).str.strip()
    d = d[(c.str.len() == 3) & c.str.upper().str.startswith("R")].copy()  # 2. NHS trusts only
    ledger["trusts"] += len(d)
    d = d[d["Treatment Function Code"] != "C_999"]          # 3. drop aggregate specialty
    ledger["no999"] += len(d)
    d["demand"] = pd.to_numeric(d["Total All"], errors="coerce")          # numeric demand
    g = (d.groupby(["Provider Org Code", "Provider Org Name",
                    "Provider Parent Org Code",
                    "Treatment Function Code", "Treatment Function Name"],
                   as_index=False)["demand"].sum())         # 4. sum over commissioner split
    g.insert(0, "date", parse_period(d["Period"].iloc[0]))  # stamp the month from Period
    ledger["canon"] += len(g)                               # canonical rows produced
    return g

print("loader defined | usecols:", len(USECOLS), "| sample parse:",
      parse_period("RTT-APRIL-2020").date(), "/", parse_period("RTT-March-2026").date())

In [ ]:
# Stage 2b · C2 — build the canonical table across all 96 months (heavy pass, ~minutes).
#   The ledger is the accountability trail: how many rows each filter step drops, and why.
ledger = dict(raw=0, part3=0, trusts=0, no999=0, canon=0)
parts = [load_month(f, ledger) for f in files]              # one canonical frame per month
canon = (pd.concat(parts, ignore_index=True)                # stack all 96 months
           .drop_duplicates(["date", "Provider Org Code",   # dedupe any revised re-releases
                             "Treatment Function Code"], keep="last")
           .rename(columns={"Provider Org Code": "branch",  # -> canonical column names
                            "Provider Org Name": "branch_name",
                            "Provider Parent Org Code": "icb",
                            "Treatment Function Code": "dept",
                            "Treatment Function Name": "dept_name"})
           .sort_values(["date", "branch", "dept"]).reset_index(drop=True))

for k, v in ledger.items():
    print(f"{k:>7}: {v:,}")                                 # raw -> part3 -> trusts -> no999 -> canon
print("dedup removed:", ledger["canon"] - len(canon))       # revised re-releases (expect 0 / small)
print("final canon rows:", f"{len(canon):,}")

In [ ]:
# Stage 2b · C3 — sanity asserts on the finished table (hard-fail if any invariant breaks).
print("rows:", f"{len(canon):,}", "| months:", canon.date.nunique(),
      "|", canon.date.min().date(), "->", canon.date.max().date())
print("trusts:", canon.branch.nunique(), "| depts:", canon.dept.nunique(),
      "| ICBs:", canon.icb.nunique())

assert canon.date.nunique() == 96                            # every month present
assert canon.duplicated(["date", "branch", "dept"]).sum() == 0   # one row per key
assert (canon.demand.dropna() >= 0).all()                    # no negative demand
print("asserts passed: 96 months, unique keys, non-negative demand")

In [ ]:
# Stage 2b · C4 — re-validate national totals, now from the FINISHED canonical table
#   (not the raw files). Proves the whole build pipeline preserved the demand signal.
EXPECT = {"2018-04": 1_544_011, "2020-04": 474_295,
          "2021-04": 1_394_578, "2026-03": 1_783_457}
for m, e in EXPECT.items():
    tot = int(canon.loc[canon.date == pd.Timestamp(m + "-01"), "demand"].sum())
    print(m, f"{tot:,}", "OK" if tot == e else "MISMATCH")

In [ ]:
# Stage 2b · C5 — save the canonical table. This ~few-MB parquet is the single source
#   every later stage loads; the 7.7 GB of raw CSVs are never touched again.
PARQUET = OUT / "rtt_canonical.parquet"
canon.to_parquet(PARQUET)
print("saved:", PARQUET, f"({PARQUET.stat().st_size / 1e6:.1f} MB)")
print("columns:", list(canon.columns))

### Stage 2b · findings  →  copy to `docs/WORKLOG.md`

- **F-09** · Filter ledger (96 months): raw **17,355,150** → `Part_3` **3,614,127** (20.8%) → trusts **2,987,812** → drop `C_999` **2,376,866** → sum commissioners → **227,344** canonical rows. **0** duplicates removed.
- **F-10** · Finished table: **227,344 rows** · 96 months (`2018-04 → 2026-03`) · **194 trusts** (union across the window) · **24 depts** · **56 ICBs**. The 56 is inflated by the **July-2022 CCG→ICB** parent-code changeover (union of old + new parent codes) — use the *latest* month's mapping for the hierarchy. National totals re-validated from the finished table **4/4**.
- **D-01 executed** → `reports/tables/rtt_canonical.parquet` (0.6 MB), columns `[date, branch, branch_name, icb, dept, dept_name, demand]`. Later stages load this, never the raw CSVs.

## Stage 3 · Missing data

**Questions:** which absences are trust **entry/exit**, which are **reporting gaps**, which are
**structural zeros** (a department a trust never offers), and what is `NaN` vs a literal `0` in demand?

Separating these species matters: entry/exit is handled by the panel choice (stage 4), gaps stay `NaN`
(the forecaster window skips them), structural zeros are simply absent rows, and a literal `0` is real data.

**Done when** `D-02` (keep / impute / drop rules per species) is logged.

In [ ]:
# Stage 3 · C1 — load the canonical table + set up figures. Every stage from here on
#   starts from the parquet (not the raw CSVs), so it is reproducible on its own.
import matplotlib.pyplot as plt
%matplotlib inline

FIG = ROOT / "reports" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
canon = pd.read_parquet(ROOT / "reports" / "tables" / "rtt_canonical.parquet")
months = pd.Index(sorted(canon.date.unique()))               # the 96-month axis
print(canon.shape, "| months:", len(months),
      "|", months[0].date(), "->", months[-1].date())

In [ ]:
# Stage 3 · C2 — trust × month presence matrix. Dark = the trust reported that month,
#   light = a gap. Horizontal light streaks reveal entry/exit and reporting gaps.
pres = (canon.groupby(["branch", "date"]).size()             # rows per trust-month
             .unstack("date").notna())                       # True where the trust reported
plt.figure(figsize=(12, 6))
plt.imshow(pres.values, aspect="auto", cmap="Greys")          # dark = present, light = absent
plt.title("trust × month presence  (light gaps = not reporting)")
plt.xlabel("month index (0 = 2018-04)"); plt.ylabel("trust index")
plt.savefig(FIG / "03_presence_heatmap.png", dpi=150, bbox_inches="tight")

print("trusts present all 96 months:", int(pres.all(axis=1).sum()), "of", pres.shape[0])

In [ ]:
# Stage 3 · C3 — which trusts entered after the window start or exited before its end.
#   These are mostly mergers/renames/reconfigurations, not data errors.
span = canon.groupby("branch").date.agg(["min", "max"])      # first & last month per trust
late = span[span["min"] > months[0]]                         # entered after 2018-04
early = span[span["max"] < months[-1]]                       # exited before 2026-03
print(len(late), "trusts enter late;", len(early), "exit early")
late.head(10)

In [ ]:
# Stage 3 · C4 — separate the two kinds of "missing" at the trust×dept (series) level.
#   Gappy series (present some months, not all) = real reporting missingness.
#   Structural zeros = trust never offers that dept -> the pair is simply absent (no row).
sd = canon.groupby(["branch", "dept"]).date.nunique()        # months present per series
full = (sd == 96).sum()                                       # complete series (all 96 months)
print(f"series: {len(sd):,} | complete: {full:,} ({full/len(sd):.0%})")

gappy = sd[(sd > 0) & (sd < 96)]                             # present sometimes = has gaps
print("gappy series:", len(gappy), "| median months present:", int(gappy.median()))

offered = canon.groupby("branch").dept.nunique()             # how many depts each trust offers
print("depts offered per trust: min", int(offered.min()),
      "median", int(offered.median()), "max", int(offered.max()), "of 24")

In [ ]:
# Stage 3 · C5 — NaN (unparseable/absent value) vs a literal 0 (a genuinely quiet month).
#   These are different things and must never be conflated: never fill NaN with 0.
print("NaN demand rows:", int(canon.demand.isna().sum()))
print("zero demand rows:", int((canon.demand == 0).sum()),
      f"({(canon.demand == 0).mean():.2%})")

### Stage 3 · findings & decision  →  copy to `docs/WORKLOG.md`

- **F-11** · Strict balanced panel: **125 of 194** trusts report in all 96 months (`reports/figures/03_presence_heatmap.png`).
- **F-12** · **11** trusts enter late, **45** exit early → exits dominate (NHS trust **mergers/consolidation**): a net loss of trusts across the window, not data gaps.
- **F-13** · **3,243** trust×dept series; **1,412 complete (44%)**, 1,831 gappy (median **60/96** months present). Trusts offer a median **20 of 24** depts → ~1,400 trust×dept pairs never appear = **structural zeros**.
- **F-14** · In the canonical table: **0 NaN** and **0 literal-zero** demand rows — every present row has demand ≥ 1. "Missingness" here is entirely *absent rows*; `NaN` appears only later when a series is reindexed to full monthly frequency (`asfreq("MS")`).

**D-02 — missingness rules:**
- trust **entry/exit** → handled by the panel choice (stage 4), *not* imputation
- **reporting gaps** inside an active series → keep as `NaN` after `asfreq("MS")`; the forecaster window skips them
- **structural zeros** (dept never offered) → absent rows, never imputed
- **literal `0`** demand → real data, kept (currently none exist)
- **`NaN`** demand cells → kept, counted, never silently filled with `0`

## Stage 4 · Characteristics

**Questions:** how **big** is demand, how **skewed**, how **spread** across trusts / depts / ICBs, and
what is the **balanced panel** at different strictness levels?

These fix two modelling facts: (1) demand is heavy-tailed and trust scale spans ~100× → **per-series
scaling** + size-aware FL weighting are mandatory; (2) the balanced panel defines the candidate
federated clients.

**Done when** `D-03` (candidate client panel) is logged.

In [ ]:
# Stage 4 · C1 — reload (keeps the stage runnable on its own) + C2 demand distribution.
#   Left: raw demand (expect heavy right skew). Right: log1p(demand) (expect roughly bell-ish,
#   maybe two humps = big vs small trusts). Implication: per-series scaling is mandatory.
canon = pd.read_parquet(ROOT / "reports" / "tables" / "rtt_canonical.parquet")
FIG = ROOT / "reports" / "figures"

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].hist(canon.demand.dropna(), bins=60); ax[0].set_title("demand")
ax[1].hist(np.log1p(canon.demand.dropna()), bins=60); ax[1].set_title("log1p(demand)")
fig.tight_layout(); fig.savefig(FIG / "04_demand_dist.png", dpi=150)

print(canon.demand.describe().round(1))
print("skew:", round(canon.demand.skew(), 2))

In [ ]:
# Stage 4 · C3 — total demand per trust, sorted, on a log axis. The spread (p90/p10 and
#   max/min) is the federated-weighting fact: clients differ ~100x in size.
tt = canon.groupby("branch").demand.sum().sort_values()      # lifetime total per trust
plt.figure(figsize=(10, 3))
plt.semilogy(range(len(tt)), tt.values)                      # log y: straight-ish line = ~exponential spread
plt.title("total demand per trust (sorted, log scale)")
plt.xlabel("trust rank"); plt.ylabel("total demand (log)")
plt.savefig(FIG / "04_trust_volume.png", dpi=150, bbox_inches="tight")

print("p90/p10 spread:", round(tt.quantile(.9) / tt.quantile(.1), 1), "x")
print("max/min:", round(tt.max() / tt.min(), 0), "x")

In [ ]:
# Stage 4 · C4 — which specialties dominate demand, and how widely each is offered.
#   Rare depts (offered by few trusts) weaken the "overlapping departments" requirement
#   for federation, so they are candidates to exclude from the client panel.
dt_ = canon.groupby("dept_name").demand.sum().sort_values(ascending=False)
print("top specialties by total demand:")
print(dt_.head(10))

cov = canon.groupby("dept").branch.nunique().sort_values(ascending=False)  # trusts offering each dept
print("\ntrusts offering each dept -- most common 3:")
print(cov.head(3))
print("...rarest 3:")
print(cov.tail(3))

In [ ]:
# Stage 4 · C5 — the balanced panel at three strictness levels. 96 = present every month
#   (strict); 90 / 84 show what relaxing the rule buys. This is a supervisor decision, so
#   present all three rather than pre-deciding.
pm = canon.groupby("branch").date.nunique()                  # months present per trust
for k in (96, 90, 84):
    print(f"trusts with >= {k} of 96 months:", int((pm >= k).sum()))

In [ ]:
# Stage 4 · C6 — trusts per ICB. The raw union of parent codes over the window is inflated
#   by the July-2022 CCG->ICB changeover, so the CORRECT hierarchy uses the latest month's
#   mapping (2026-03), where `Provider Parent` = the real ~42 ICBs.
icb_raw = canon.groupby("icb").branch.nunique()
print("raw union of parent codes over the window:", len(icb_raw), "distinct (inflated)")

latest = canon[canon.date == canon.date.max()]               # 2026-03 snapshot
icb_now = latest.groupby("icb").branch.nunique().sort_values(ascending=False)
print(len(icb_now), "ICBs in 2026-03 | trusts per ICB: min", int(icb_now.min()),
      "median", int(icb_now.median()), "max", int(icb_now.max()))
print("\nlargest ICBs by trust count:")
print(icb_now.head(10))